In [ ]:
from datetime import datetime
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image, ImageOps
from transformers import CLIPTokenizer

import model_loader
import pipeline


# =========================
# PATHS
# =========================

SD_DIR = Path.cwd()
if SD_DIR.name != "sd" and (SD_DIR / "sd").is_dir():
    SD_DIR = SD_DIR / "sd"

ROOT_DIR = SD_DIR.parent
DATA_DIR = ROOT_DIR / "data"
IMAGES_DIR = ROOT_DIR / "images"
OUTPUT_DIR = ROOT_DIR / "outputs"

IMAGES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

VOCAB_FILE = DATA_DIR / "vocab.json"
MERGES_FILE = DATA_DIR / "merges.txt"
MODEL_FILE = DATA_DIR / "v1-5-pruned-emaonly.ckpt"


# =========================
# DEVICE
# =========================

RUN_DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
IDLE_DEVICE = torch.device("cpu")

print(f"Running inference on: {RUN_DEVICE}")


# =========================
# TOKENIZER AND MODEL
# =========================

tokenizer = CLIPTokenizer(
    vocab=str(VOCAB_FILE),
    merges=str(MERGES_FILE),
)

models = model_loader.preload_models_from_standard_weights(
    str(MODEL_FILE),
    IDLE_DEVICE,
)


# =========================
# IMAGE SIZE
# =========================

pipeline.WIDTH = 512
pipeline.HEIGHT = 512
pipeline.LATENTS_WIDTH = pipeline.WIDTH // 8
pipeline.LATENTS_HEIGHT = pipeline.HEIGHT // 8

print("Model ready.")

In [ ]:
# Image to Image

input_path = IMAGES_DIR / "dog.png"

input_image = Image.open(input_path).convert("RGB")

# Crop dan resize tanpa membuat gambar gepeng.
input_image = ImageOps.fit(
    input_image,
    (pipeline.WIDTH, pipeline.HEIGHT),
    method=Image.Resampling.LANCZOS,
)

prompt = (
    "a realistic photograph of the same dog, "
    "wearing black sunglasses directly over both eyes, "
    "the lenses covering both eyes, "
    "the frame resting on the bridge of its nose, "
    "natural anatomy, cinematic lighting, highly detailed"
)

output_image = pipeline.generate(
    prompt=prompt,
    uncond_prompt="",

    input_image=input_image,
    strength=0.65,

    do_cfg=True,
    cfg_scale=7.5,

    sampler_name="ddpm",
    n_inference_steps=30,
    seed=42,

    models=models,
    device=RUN_DEVICE,
    idle_device=IDLE_DEVICE,
    tokenizer=tokenizer,
)

result = Image.fromarray(output_image)

output_path = (
    OUTPUT_DIR
    / f"img2img_{datetime.now():%Y%m%d_%H%M%S}.png"
)
result.save(output_path)

print(f"Image saved to: {output_path}")

print("Input:")
display(input_image)

print("Output:")
display(result)

In [ ]:
# TEXT-TO-IMAGE

prompt = (
    "a realistic full-body photograph of one domestic short-haired cat "
    "lying naturally on a tiled floor, four clearly separated legs, "
    "a distinct tail curled behind its body, natural feline anatomy, "
    "soft cinematic lighting, highly detailed"
)

output_image = pipeline.generate(
    prompt=prompt,
    uncond_prompt="",

    input_image=None,
    strength=0.9,

    do_cfg=True,
    cfg_scale=7.5,

    sampler_name="ddpm",
    n_inference_steps=10,
    seed=42,

    models=models,
    device=RUN_DEVICE,
    idle_device=IDLE_DEVICE,
    tokenizer=tokenizer,
)

result = Image.fromarray(output_image)

output_path = (
    OUTPUT_DIR
    / f"txt2img_{datetime.now():%Y%m%d_%H%M%S}.png"
)

result.save(output_path)

print(f"Image saved to: {output_path}")
display(result)